# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    LIMIT 100
""").df()

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
df.sample(1)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
28,2026-03-01,client_73cda7b4e4f265ea,content_419dc7d89aee9698,True,False,True,<NA>,50,0,318,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


Unit of analysis: one row = one content item (page), belonging to one content.

Time window: month = 2026-03 — a single mid-panel month, chosen deliberately to avoid the _sample table (which is the sealed final month, June 2026, reserved only for testing query mechanics, not for developing label logic) and to keep an honest distance from any period I might later use as a test window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               100 non-null    datetime64[us]
 1   client_hash_id            100 non-null    str           
 2   content_hash_id           100 non-null    str           
 3   client_has_gsc            100 non-null    bool          
 4   client_has_ga4            100 non-null    bool          
 5   gsc_data_available        100 non-null    bool          
 6   ga4_data_available        0 non-null      boolean       
 7   gsc_impressions           100 non-null    int64         
 8   gsc_clicks                100 non-null    int64         
 9   gsc_sum_position          100 non-null    int64         
 10  gsc_avg_position          79 non-null     float64       
 11  ga4_pageviews             0 non-null      Int64         
 12  ga4_sessions              0 non-nu

Feature: gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, gsc_data_available — the populated Search Console signals directly relevant to Ranking Signal Analysis.

Label: No ready-made target exists in this table (unlike the Week 1–2 starter CSV's trend_direction) — I'll need to derive a label myself, likely from a rolling comparison of gsc_avg_position or gsc_clicks across dates.

Context: report_date, client_hash_id, content_hash_id, month — identify the row (my stated grain) but aren't behavioral signals themselves.

Excluded: all GA4 and AI-referral columns (ga4_*, sessions_*, ai_*, scroll_events) — every one of these shows 0 non-null values in this slice, meaning this client/month combination has no GA4 or AI-referral data available. Excluded because there's nothing to learn from an entirely empty column.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print('This verifies that the data the unit of analysis: one row = one content item.')
df['content_hash_id'].nunique()

This verifies that the data the unit of analysis: one row = one content item.


100

In [12]:
print('This verifies that the data only spans a specific time window')

df['report_date'].nunique()

This verifies that the data only spans a specific time window


1

In [8]:
df.sample(5)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
42,2026-03-01,client_73cda7b4e4f265ea,content_a2dc03831230c20e,True,False,True,<NA>,50,0,492,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
40,2026-03-01,client_73cda7b4e4f265ea,content_a131ba36ceccfb86,True,False,True,<NA>,47,0,618,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
49,2026-03-01,client_73cda7b4e4f265ea,content_4cd532e5e9edc02b,True,False,True,<NA>,205,0,745,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
91,2026-03-01,client_73cda7b4e4f265ea,content_e3c559b94cfdd811,True,False,True,<NA>,2,0,18,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
30,2026-03-01,client_73cda7b4e4f265ea,content_8935ed68eca88b01,True,False,True,<NA>,113,0,639,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell me anything about GA4 engagement or AI-referral traffic — every ga4_*, sessions_*, and ai_* column is entirely empty (0 non-null) for this client/month, so any question involving pageviews, sessions, or AI-driven traffic is simply unanswerable from this data. The data is also GSC-only in practice: my usable signals are limited to gsc_impressions, gsc_clicks, gsc_sum_position, and gsc_avg_position, and even gsc_avg_position has gaps (79/100 non-null in my sample), so some rows carry impressions/clicks but no position value. This single month also can't show seasonal or long-term trend behavior — one month is a snapshot, not a trajectory, so I can't distinguish a genuine decline from normal month-to-month noise without comparing against neighboring months. Finally, if my full slice turns out to represent very few clients (as my sample suggested), any pattern I find may reflect that client's specific behavior rather than a generalizable signal across clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ Some code cells are filled, some are not ] Every section above is filled — markdown thinking AND the code that backs it
- [ * ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ * ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ * ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.